<center><h1>Blombach_George_HW7</h1></center>

Name: George Blombach | Github Username: gblombach | USC ID: 2416-0961-99

## 1. Multi-class and Multi-Label Classification Using Support Vector Machines

Import packages

In [1]:
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.svm import SVC, LinearSVC
from sklearn.base import clone
from sklearn.metrics import hamming_loss, accuracy_score

### (a) Download the Anuran Calls (MFCCs) Data Set

In [2]:
#7,195 observations, 22 MFCC predictors, with 4 families, 8 genera, and 10 species
csv_data = pd.read_csv('../data/Frogs_MFCCs.csv', na_values='na')
display(csv_data.shape)
#display(csv_data.isna().sum().to_frame("Missing Values")) #check for missing values
labels = ["Family", "Genus", "Species"]

(7195, 26)

In [3]:
X = csv_data.iloc[:,:22].copy()                          # get MFCC Features
y = csv_data[labels].copy()      # get labels 
X_train, X_test, y_train, y_test = train_test_split(X, y, train_size = 0.7, shuffle=True, random_state=333)
X_train = X_train.reset_index(drop=True)                 # Reset indices 
X_test = X_test.reset_index(drop=True)
y_train = y_train.reset_index(drop=True)
y_test = y_test.reset_index(drop=True)
display(X_train.shape, X_test.shape, y_train.shape, y_test.shape)

(5036, 22)

(2159, 22)

(5036, 3)

(2159, 3)

### (b) Train a classifier for each label

#### (i) Research

Exact match and Hamming loss are two methods that can be used to evaluate multi-label classification:
- **Exact match**, also referred to as subset accuracy, requires the entire predicted set of labels for an observation to match the true set of labels. Consequently, it is a strict measure because an observation is considered incorrectly classified when even one of its labels is predicted incorrectly.
- **Hamming loss** evaluates predictions at the individual-label level and measures the fraction of instance-label pairs that are misclassified. Thus, unlike exact match, Hamming loss distinguishes between predictions in which only one label is incorrect and predictions in which all labels are incorrect.

For exact match, a higher score is better, with 1 representing perfect classification, while for Hamming loss a lower value is better, with 0 representing no incorrectly classified labels.References:

- Tsoumakas, G., Katakis, I., & Vlahavas, I. (2010). “Mining Multi-label Data.” In *Data Mining and Knowledge Discovery Handbook*, 2nd ed., Springer.
- Zhang, M.-L., & Zhou, Z.-H. (2014). “A Review on Multi-Label Learning Algorithms.” *IEEE Transactions on Knowledge and Data Engineering*, 26(8), 1819–1837.


#### (ii) Train a SVM for each of the labels

In [4]:
#Evaluation function
def evaluate_model(y_true, y_pred, model):
    exact_match = (y_true[labels].to_numpy()  == y_pred[labels].to_numpy()).all(axis=1).mean() #Exact match: True only when Family, Genus, AND Species are all correct
    hamming = (y_true[labels].to_numpy() != y_pred[labels].to_numpy()).mean() # Hamming loss: Fraction of individual labels predicted incorrectly
    hamming_score = 1 - hamming # Hamming score defined here as 1 - Hamming loss
    results = {"Model": model, "Exact Match": exact_match, "Hamming Loss": hamming, "Hamming Score": hamming_score}

    for label in labels:  # accuracy_score is fine here because each call contains only ONE multiclass output
        results[f"{label} Accuracy"] = accuracy_score(y_true[label], y_pred[label])
    return results
    
def evauate_models(y_true, y_pred, model):
    exact_match = np.mean(np.all(y_true.to_numpy() == y_pred.to_numpy(), axis=1)) # Exact match: is correct only if ALL three labels are correct
    hamming = hamming_loss(y_true, y_pred)  # Hamming loss: Fraction of individual labels predicted incorrectly
    hamming_score = 1 - hamming
    results = {"Model": model,"Exact Match": exact_match,"Hamming Loss": hamming, "Hamming Score": hamming_score}
    return results

cv = StratifiedKFold(n_splits=10, shuffle=True,random_state=333)                        #10-fold cross-validation.
C_values = [0.001, 0.01, 0.1, 1.0, 10.0, 100.0, 1000.0, 10000.0, 100000.0, 1000000.0]   #C controls the penalty for misclassification
sigma_values = np.arange(0.1, 2.1, 0.1)                                                 #sigma .1 to 2.0
gamma_values = 1 / (2 * sigma_values ** 2)                                              #gamma

# Tuning function
def tune_classifier(estimator, parameter_grid, X_train, y_train, X_test, model):
    predictions = pd.DataFrame(index=range(len(X_test)), columns=labels)
    best_parameters = {}; best_cv_scores = {}; fitted_searches = {}
    for label in labels:
        print(f"\nTuning {model} for {label}...")
        grid_search = GridSearchCV(estimator=clone(estimator), param_grid=parameter_grid, scoring="accuracy", cv=cv, n_jobs=-1, refit=True)
        grid_search.fit(X_train, y_train[label])
        predictions[label] = grid_search.predict(X_test)
        best_parameters[label] = grid_search.best_params_
        best_cv_scores[label] = grid_search.best_score_
        fitted_searches[label] = grid_search
        best_params = {key: float(value) if isinstance(value, np.floating) else value for key, value in grid_search.best_params_.items()}
        print("Best parameters:", best_params, "\nBest mean CV accuracy:", round(grid_search.best_score_, 4))
    return (predictions, best_parameters, best_cv_scores, fitted_searches)

gaussian_svm = SVC(kernel="rbf", decision_function_shape="ovr")
gaussian_parameter_grid = {"C": C_values,"gamma": gamma_values}
(gaussian_predictions, gaussian_best_parameters, gaussian_cv_scores, gaussian_searches) = tune_classifier(estimator=gaussian_svm, 
            parameter_grid=gaussian_parameter_grid, X_train=X_train, y_train=y_train, X_test=X_test, model="Gaussian SVM")

gaussian_parameter_rows = []

for label in labels:
    best_C = gaussian_best_parameters[label]["C"]
    best_gamma = gaussian_best_parameters[label]["gamma"]
    best_sigma = np.sqrt(1 / (2 * best_gamma))
    gaussian_parameter_rows.append({"Label": label, "Best C": best_C, "Best Gamma": best_gamma, "Best Sigma": best_sigma,
                                    "Best CV Accuracy": gaussian_cv_scores[label]})
gaussian_parameter_table = pd.DataFrame(gaussian_parameter_rows)
display(gaussian_parameter_table.style.format({"Best C": "{:.6g}", "Best Gamma": "{:.6g}", "Best Sigma": "{:.4f}", "Best CV Accuracy": "{:.4f}" }).hide(axis="index"))
gaussian_results = evaluate_model(y_true=y_test, y_pred=gaussian_predictions, model="Gaussian SVM Binary Relevance")
display(pd.DataFrame([gaussian_results]).style.format({column: "{:.4f}" for column in gaussian_results if column != "Model"}).hide(axis="index"))


Tuning Gaussian SVM for Family...
Best parameters: {'C': 100.0, 'gamma': 2.0} 
Best mean CV accuracy: 0.9934

Tuning Gaussian SVM for Genus...
Best parameters: {'C': 10.0, 'gamma': 3.1249999999999996} 
Best mean CV accuracy: 0.9913

Tuning Gaussian SVM for Species...
Best parameters: {'C': 10.0, 'gamma': 2.0} 
Best mean CV accuracy: 0.9905


Label,Best C,Best Gamma,Best Sigma,Best CV Accuracy
Family,100,2,0.5000,0.9934
Genus,10,3.125,0.4000,0.9913
Species,10,2,0.5000,0.9905


Model,Exact Match,Hamming Loss,Hamming Score,Family Accuracy,Genus Accuracy,Species Accuracy
Gaussian SVM Binary Relevance,0.9880,0.0063,0.9937,0.9944,0.9940,0.9926


#### (iii) Repeat 1(b)ii with L1-penalized SVMs

#### (iv) Repeat 1(b)iii by using SMOTE or any other method for imbalance

## 2. K-Means Clustering on a Multi-Class and Multi-Label Data Set

### (a) Use k-means clustering

### (b) Determine which family is the majority

### (c) Calculate the average Hamming distance, Hamming score, and Hamming loss

## 3. ISLR 12.6.2 (p. 546)